# Fairness audit of a loan model
**Veritas AI · International High School Hackathon 2026 · Project 4.2.3**

**Goal:** build a model that decides who gets a loan, then test whether it treats young applicants (under 25) differently from everyone else, and work out why.

| Section | What it does |
|---|---|
| 0 | Setup |
| 1 | Data, the 5:1 cost rule, three models compared against baselines |
| 2 | Fairness: approval rates for under-25s vs 25+, the gap, and how sure we are |
| 3 | What drives the gap |
| 4 | Proxy test: does deleting age remove the bias? |
| 5 | Fixing the gap, and what it costs |
| 6 | Flip test, an audit of the data itself, and deeper fairness checks |
| 7 | Tail risk: losses by loan size, recessions and worst cases |
| 8 | Interactive demo |
| 9 | Limitations and how to overcome them |

*How to use this notebook in Colab: **Runtime → Run all**. Everything downloads automatically.*

## 0. Setup

In [ ]:
!pip install -q ucimlrepo fairlearn

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

SEED = 42                      # fixed so everyone gets the same numbers
rng = np.random.default_rng(SEED)
pd.set_option("display.precision", 3)

# Chart colours (colour-blind safe pair)
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8984"
MODEL_COLOURS = {"Logistic regression": BLUE, "Random forest": ORANGE, "Gradient boosting": "#1baf7a"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

## 1. Data and models

### 1.1 Load the data
The Statlog German Credit dataset: 1,000 loans from a bank in southern Germany, 1973–75, 20 details each, labelled **good risk** (paid back) or **bad risk**.

The UCI version stores categories as codes like `A11`, so we translate them into plain English.

In [ ]:
COLS = ["status", "duration", "credit_history", "purpose", "amount", "savings",
        "employment_duration", "installment_rate", "personal_status_sex", "other_debtors",
        "present_residence", "property", "age", "other_installment_plans", "housing",
        "number_credits", "job", "people_liable", "telephone", "foreign_worker"]

# Translation of UCI codes (from the dataset documentation)
CODES = {
 "A11": "< 0 DM", "A12": "0-200 DM", "A13": ">= 200 DM", "A14": "no checking account",
 "A30": "no credits / all paid", "A31": "all paid at this bank", "A32": "existing paid duly",
 "A33": "past delays", "A34": "critical account",
 "A40": "car (new)", "A41": "car (used)", "A42": "furniture", "A43": "radio/TV",
 "A44": "appliances", "A45": "repairs", "A46": "education", "A47": "vacation",
 "A48": "retraining", "A49": "business", "A410": "other",
 "A61": "< 100 DM", "A62": "100-500 DM", "A63": "500-1000 DM", "A64": ">= 1000 DM", "A65": "unknown / none",
 "A71": "unemployed", "A72": "< 1 year", "A73": "1-4 years", "A74": "4-7 years", "A75": ">= 7 years",
 # Personal status: the original UCI labels are wrong (see section 6.2), so we use neutral names
 "A91": "status code 1", "A92": "status code 2 (married women AND single men)",
 "A93": "status code 3", "A94": "status code 4", "A95": "status code 5 (unused)",
 "A101": "none", "A102": "co-applicant", "A103": "guarantor",
 "A121": "real estate", "A122": "savings / life insurance", "A123": "car / other", "A124": "none",
 "A141": "bank", "A142": "stores", "A143": "none",
 "A151": "rent", "A152": "own", "A153": "for free",
 "A171": "unemployed / unskilled non-resident", "A172": "unskilled resident",
 "A173": "skilled", "A174": "management / self-employed",
 "A191": "no", "A192": "yes", "A201": "yes", "A202": "no",
}
BACKUP_URL = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
# The backup copy uses its own category wording, so we convert it back to the official UCI codes
BACKUP_TO_UCI = {"status": {"... < 100 DM": "A11", "... >= 200 DM / salary for at least 1 year": "A13", "0 <= ... < 200 DM": "A12", "no checking account": "A14"}, "credit_history": {"all credits at this bank paid back duly": "A31", "critical account/other credits existing": "A34", "delay in paying off in the past": "A33", "existing credits paid back duly till now": "A32", "no credits taken/all credits paid back duly": "A30"}, "purpose": {"business": "A48", "car (new)": "A40", "car (used)": "A41", "domestic appliances": "A43", "education": "A45", "furniture/equipment": "A410", "others": "A49", "radio/television": "A42", "repairs": "A44", "retraining": "A46"}, "savings": {"... < 100 DM": "A61", "... >= 1000 DM": "A64", "100 <= ... < 500 DM": "A62", "500 <= ... < 1000 DM": "A63", "unknown/no savings account": "A65"}, "employment_duration": {"... < 1 year": "A72", "... >= 7 years": "A75", "1 <= ... < 4 years": "A73", "4 <= ... < 7 years": "A74", "unemployed": "A71"}, "personal_status_sex": {"female : divorced/separated/married": "A92", "male : divorced/separated": "A91", "male : married/widowed": "A94", "male : single": "A93"}, "other_debtors": {"co-applicant": "A102", "guarantor": "A103", "none": "A101"}, "property": {"building society savings agreement/life insurance": "A122", "car or other": "A123", "real estate": "A121", "unknown/no property": "A124"}, "other_installment_plans": {"bank": "A141", "none": "A143", "stores": "A142"}, "housing": {"for free": "A153", "own": "A152", "rent": "A151"}, "job": {"management/self-employed/highly qualified employee/officer": "A174", "skilled employee/official": "A173", "unemployed/unskilled - non-resident": "A171", "unskilled - resident": "A172"}, "telephone": {"no": "A191", "yes": "A192"}, "foreign_worker": {"no": "A202", "yes": "A201"}}

def from_uci_codes(X, target):
    """Official UCI format (codes like A11, target 1 = good, 2 = bad) -> readable table."""
    X = X.copy()
    X.columns = COLS
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].astype(str).map(lambda v: CODES.get(v, v))
    y = (pd.Series(target).astype(int).to_numpy() == 1).astype(int)
    return X.reset_index(drop=True), pd.Series(y, name="good")

def load_data():
    """1) UCI via ucimlrepo, as the brief asks  2) a local german.data file  3) a GitHub backup copy."""
    try:
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=144)
        X = d.data.features
        if all(str(c).startswith("Attribute") for c in X.columns):   # make sure Attribute1..20 are in order
            X = X[sorted(X.columns, key=lambda c: int(str(c).replace("Attribute", "")))]
        return (*from_uci_codes(X, d.data.targets.iloc[:, 0]), "UCI, fetch_ucirepo(id=144)")
    except Exception as e:
        print("UCI download failed:", e)
    import os
    if os.path.exists("german.data"):
        raw = pd.read_csv("german.data", sep=" ", header=None)
        return (*from_uci_codes(raw.iloc[:, :20], raw.iloc[:, 20]), "local german.data file")
    raw = pd.read_csv(BACKUP_URL)
    target = raw.pop("credit_risk").map({1: 1, 0: 2})                   # backup: 1 = good, 0 = bad
    X = raw[COLS].copy()
    for c, mapping in BACKUP_TO_UCI.items():
        X[c] = X[c].map(mapping)
    return (*from_uci_codes(X, target), "GitHub backup copy (converted to UCI codes)")

X, y, SOURCE = load_data()
young = (X["age"] < 25).to_numpy()            # the group we audit

print(f"Source: {SOURCE}")
print(f"{len(X):,} applicants, {X.shape[1]} details each")
print(f"Good risks: {y.sum()}   Bad risks: {(1 - y).sum()}")
print(f"Under 25: {young.sum()}   25 or older: {(~young).sum()}")
X.head()

**First look:** how often were people in each age group actually good risks? This matters later, because if young people really did repay less often, some gap in approvals may be expected. The question is whether the model's gap is *bigger* than the real difference.

In [ ]:
first_look = pd.DataFrame({
    "applicants": [young.sum(), (~young).sum()],
    "actually good risk": [y[young].mean(), y[~young].mean()],
}, index=["Under 25", "25 or older"])
first_look.style.format({"actually good risk": "{:.1%}"})

### 1.2 The 5:1 cost rule, and the best threshold
The dataset's documentation says **approving a bad risk is 5 times as costly as rejecting a good one**:

- reject a good applicant → cost 1 (a lost customer)
- approve a bad applicant → cost 5 (an unpaid loan)

Our model gives each applicant a probability $p$ of being a good risk. Approving them has expected cost

$$5(1-p)$$

and rejecting them has expected cost

$$1 \times p$$

So we should approve only when $5(1-p) < p$, which simplifies to

$$p > \frac{5}{6}$$

**So the model needs to be about 83% sure before it lends.** This one number turns out to matter a lot for fairness, because it means the bank turns away many borderline applicants, and young applicants are more likely to be borderline.

**Two baselines to beat:**

In [ ]:
THRESHOLD = 5 / 6

def evaluate(y_true, approve):
    y_true, approve = np.asarray(y_true), np.asarray(approve).astype(bool)
    bad_approved = int(((y_true == 0) & approve).sum())
    good_rejected = int(((y_true == 1) & ~approve).sum())
    return {"accuracy": float((approve == (y_true == 1)).mean()),
            "cost": 5 * bad_approved + 1 * good_rejected,
            "approval rate": float(approve.mean()),
            "bad risks approved": bad_approved,
            "good risks rejected": good_rejected}

baselines = pd.DataFrame({
    "Approve everyone": evaluate(y, np.ones(len(y))),
    "Reject everyone": evaluate(y, np.zeros(len(y))),
}).T
baselines

**Surprise:** rejecting *everyone* (cost 700) is cheaper than approving everyone (cost 1,500). So beating "approve everyone" is easy. The real test is whether a model beats **700**. Most teams will miss this.

### 1.3 Three models, tested on data they have never seen
Each model is tested with **5-fold cross-validation**: train on 800 applicants, predict the hidden 200, repeat 5 times so every applicant is predicted exactly once while unseen.

Inside each training set, a second, smaller cross-validation picks the regularisation strength (how simple the model is forced to be). This is called *nested* cross-validation; it stops the test data leaking into those choices.

| Model | Why it's here | Regularisation we tune |
|---|---|---|
| Logistic regression | Simple and explainable: one weight per detail | `C` (smaller = simpler) |
| Random forest | Many decision trees voting | minimum leaf size, features per split |
| Gradient boosting | Usually the most accurate | tree depth, learning rate, L2 penalty |

In [ ]:
CAT = X.select_dtypes("object").columns.tolist()
NUM = X.select_dtypes("number").columns.tolist()

def make_preprocessor(cat, num):
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
        ("num", StandardScaler(), num)])

def make_models(cat=CAT, num=NUM):
    pre = lambda: make_preprocessor(cat, num)
    return {
        "Logistic regression": (
            Pipeline([("prep", pre()), ("model", LogisticRegression(max_iter=5000))]),
            {"model__C": [0.01, 0.03, 0.1, 0.3, 1, 3]}),
        "Random forest": (
            Pipeline([("prep", pre()), ("model", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]),
            {"model__min_samples_leaf": [1, 5, 10], "model__max_features": ["sqrt", 0.3]}),
        "Gradient boosting": (
            Pipeline([("prep", pre()), ("model", HistGradientBoostingClassifier(max_iter=200, random_state=SEED))]),
            {"model__max_depth": [2, 3], "model__learning_rate": [0.03, 0.1], "model__l2_regularization": [0, 1]}),
    }

OUTER = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
INNER = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)

def out_of_fold_probs(pipe, grid, X, y, sample_weight=None):
    # Probability of 'good' for every applicant, each predicted while hidden from training.
    probs, folds, chosen = np.zeros(len(y)), np.zeros(len(y), dtype=int), []
    for k, (tr, te) in enumerate(OUTER.split(X, y)):
        search = GridSearchCV(pipe, grid, cv=INNER, scoring="neg_log_loss", n_jobs=-1)
        fit_args = {} if sample_weight is None else {"model__sample_weight": np.asarray(sample_weight)[tr]}
        search.fit(X.iloc[tr], y.iloc[tr], **fit_args)
        probs[te] = search.predict_proba(X.iloc[te])[:, 1]
        folds[te] = k
        train_acc = ((search.predict_proba(X.iloc[tr])[:, 1] > 0.5) == y.iloc[tr].to_numpy()).mean()
        chosen.append({"settings": search.best_params_, "train accuracy": train_acc})
    return probs, folds, chosen

oof, train_acc = {}, {}
for name, (pipe, grid) in make_models().items():
    probs, folds, chosen = out_of_fold_probs(pipe, grid, X, y)
    oof[name] = probs
    train_acc[name] = np.mean([c["train accuracy"] for c in chosen])
    print(f"{name}: done. Settings picked in each fold: {[c['settings'] for c in chosen]}")
FOLDS = folds   # same folds for every model

### 1.4 Results: accuracy, cost and approvals
"±" shows how much the score varied between the 5 folds. Cost is scaled to 1,000 applicants so it's comparable with the baselines.

In [ ]:
rows = {}
for name, p in oof.items():
    approve = p > THRESHOLD
    overall = evaluate(y, approve)
    per_fold = pd.DataFrame([evaluate(y[FOLDS == k], approve[FOLDS == k]) for k in range(5)])
    rows[name] = {
        "AUC": roc_auc_score(y, p),
        "accuracy": overall["accuracy"],
        "accuracy ± (fold sd)": per_fold["accuracy"].std(),
        "cost": overall["cost"],
        "cost ± (fold sd, x5)": per_fold["cost"].std() * 5,
        "approval rate": overall["approval rate"],
        "accuracy at 50% threshold": evaluate(y, p > 0.5)["accuracy"],
        "cost at 50% threshold": evaluate(y, p > 0.5)["cost"],
    }
results = pd.DataFrame(rows).T
for b in baselines.index:
    results.loc[b, ["accuracy", "cost", "approval rate"]] = baselines.loc[b, ["accuracy", "cost", "approval rate"]].astype(float)
results.style.format({c: "{:.3f}" for c in ["AUC", "accuracy", "accuracy ± (fold sd)", "approval rate", "accuracy at 50% threshold"]}
                     | {c: "{:,.0f}" for c in ["cost", "cost ± (fold sd, x5)", "cost at 50% threshold"]}, na_rep="–")

**Overfitting check: in-sample vs out-of-sample.** Accuracy on the 800 applicants each model *trained on*, against accuracy on the 200 it *never saw* (both at a 50% threshold). A big drop means the model memorised its training data instead of learning general patterns.

In [ ]:
fit_check = pd.DataFrame({
    "in-sample accuracy (training data)": train_acc,
    "out-of-sample accuracy (unseen data)": {n: evaluate(y, p > 0.5)["accuracy"] for n, p in oof.items()},
})
fit_check["drop"] = fit_check.iloc[:, 0] - fit_check.iloc[:, 1]
fit_check.style.format("{:.1%}")

**Chart: cost at every possible threshold.** Lower is better. The dashed line marks the theoretical best threshold of 5/6. The grey lines are the two baselines.

In [ ]:
grid_t = np.linspace(0.05, 0.98, 120)
fig, ax = plt.subplots(figsize=(8, 4.2))
for name, p in oof.items():
    costs = [evaluate(y, p > t)["cost"] for t in grid_t]
    ax.plot(grid_t, costs, lw=2, color=MODEL_COLOURS[name], label=name)
ax.axhline(1500, color=GREY, lw=1, ls=":"); ax.text(0.06, 1530, "Approve everyone (1,500)", color=GREY, fontsize=9)
ax.axhline(700, color=GREY, lw=1, ls=":");  ax.text(0.06, 730, "Reject everyone (700)", color=GREY, fontsize=9)
ax.axvline(THRESHOLD, color="#52514e", lw=1, ls="--"); ax.text(THRESHOLD - 0.01, 1450, "5/6", ha="right", fontsize=9)
ax.set_xlabel("Approve if the model is at least this sure the applicant is a good risk")
ax.set_ylabel("Total cost (5 per bad risk approved, 1 per good rejected)")
ax.set_title("Models only beat 'reject everyone' when they are strict about lending", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper center")
plt.tight_layout(); plt.show()

**Accuracy and cost pull in opposite directions.** At a 50% threshold the models beat approve-everyone on accuracy (about 75% vs 70%), but they cost far more than rejecting everyone. At the cost-optimal 5/6 threshold, cost drops well below 700, but accuracy falls *below* 70%, because the model now turns away many good applicants to avoid the expensive bad ones. **Which one is "better" depends on what the bank cares about**, so we report both.

### 1.5 Pick a model
We pick the model with the **lowest cost** for now. In section 2 we check whether it is also fair. If a slightly more expensive model is much fairer, that's a trade-off worth discussing in the presentation.

In [ ]:
model_rows = results.loc[list(oof)]
BEST = model_rows["cost"].astype(float).idxmin()
p_best = oof[BEST]
approve_best = p_best > THRESHOLD
print(f"Chosen model: {BEST}")
print(f"  accuracy {results.loc[BEST, 'accuracy']:.1%}, cost {results.loc[BEST, 'cost']:,.0f} "
      f"(vs 700 for reject-everyone, 1,500 for approve-everyone)")

## 2. Fairness: does the model treat under-25s differently?

We measure three things for each age group:

1. **Approval rate**: what share of the group gets a loan.
2. **Approval rate among good risks** (*equal opportunity*, **our main fairness measure**): of the people who *really did* pay back, what share got a loan? If this differs, the model is turning away reliable young people more often. We lead with this because it asks exactly our question (are creditworthy under-25s penalised?) and, unlike equal approval rates, it doesn't punish a model for real differences in repayment. Its weakness: it trusts the good/bad labels, which come from past bank decisions (section 6.2).
3. **Actual good-risk rate**: what really happened. This shows how much of the gap reflects real differences in repayment.

We also show the **ratio** of approval rates. A ratio below **0.8** is the "four-fifths rule": a US rule of thumb from employment law, used here as a yardstick, **not a UK legal test**.

Because only about 150 applicants are under 25, any gap could partly be luck. **Bootstrapping** handles this: we resample the applicants 2,000 times, recompute the gap each time, and the middle 95% of results gives a confidence interval.

In [ ]:
def group_metrics(approve, y, young):
    y, approve = np.asarray(y), np.asarray(approve).astype(bool)
    out = {}
    for label, g in [("Under 25", young), ("25 or older", ~young)]:
        out[label] = {"approval rate": approve[g].mean(),
                      "approval rate among good risks": approve[g & (y == 1)].mean(),
                      "actual good-risk rate": y[g].mean()}
    return pd.DataFrame(out).T

def bootstrap_gaps(approve, y, young, n=2000, seed=SEED):
    # 95% intervals for the gaps (25+ minus under-25) and for the approval ratio.
    r = np.random.default_rng(seed)
    y, approve = np.asarray(y), np.asarray(approve).astype(float)
    idx = r.integers(0, len(y), size=(n, len(y)))
    A, Y, G = approve[idx], y[idx], young[idx]
    def rate(mask, vals):
        return (vals * mask).sum(1) / mask.sum(1)
    old_app, yng_app = rate(~G, A), rate(G, A)
    old_eo, yng_eo = rate(~G & (Y == 1), A), rate(G & (Y == 1), A)
    stats = {"approval gap": old_app - yng_app,
             "good-risk approval gap": old_eo - yng_eo,
             "approval ratio (young / older)": yng_app / old_app}
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in stats.items()}

def fairness_summary(approve):
    gm = group_metrics(approve, y, young)
    ci = bootstrap_gaps(approve, y, young)
    point = {"approval gap": gm.loc["25 or older", "approval rate"] - gm.loc["Under 25", "approval rate"],
             "good-risk approval gap": gm.loc["25 or older", "approval rate among good risks"]
                                        - gm.loc["Under 25", "approval rate among good risks"],
             "approval ratio (young / older)": gm.loc["Under 25", "approval rate"] / gm.loc["25 or older", "approval rate"]}
    return gm, point, ci

gm, point, ci = fairness_summary(approve_best)
print(f"Model: {BEST}, threshold 5/6\n")
display(gm.style.format("{:.1%}"))
for k in point:
    fmt = (lambda v: f"{v:.2f}") if "ratio" in k else (lambda v: f"{v * 100:+.1f} percentage points")
    print(f"{k:32s} {fmt(point[k])}   (95% CI {fmt(ci[k][0])} to {fmt(ci[k][1])})")

**Chart: how each age group is treated.** Error bars show the 95% bootstrap confidence interval for each rate.

In [ ]:
def rate_ci(values, mask, n=2000, seed=SEED):
    r = np.random.default_rng(seed)
    v = np.asarray(values, dtype=float)[mask]
    boots = r.choice(v, size=(n, len(v))).mean(1)
    return v.mean(), *np.percentile(boots, [2.5, 97.5])

yv = y.to_numpy()
metrics = [("Approval rate", approve_best, np.ones(len(yv), bool)),
           ("Approval rate\namong good risks", approve_best, yv == 1),
           ("Actual good-risk\nrate (reality)", yv, np.ones(len(yv), bool))]
fig, ax = plt.subplots(figsize=(8, 4.2))
w = 0.36
for j, (label, g, colour) in enumerate([("Under 25", young, ORANGE), ("25 or older", ~young, BLUE)]):
    for i, (_, vals, extra) in enumerate(metrics):
        m, lo, hi = rate_ci(vals, g & extra)
        x = i + (j - 0.5) * (w + 0.02)
        ax.bar(x, m, width=w, color=colour, label=label if i == 0 else None)
        ax.errorbar(x, m, yerr=[[m - lo], [hi - m]], color="#0b0b0b", capsize=3, lw=1)
        ax.text(x, hi + 0.02, f"{m:.0%}", ha="center", fontsize=9, color="#0b0b0b")
ax.set_xticks(range(len(metrics))); ax.set_xticklabels([m[0] for m in metrics])
ax.set_ylim(0, 1.08); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"{BEST}: under-25s are approved less often", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper right", ncol=2)
plt.tight_layout(); plt.show()

### 2.1 Is the gap bigger than reality?
Compare the **approval gap** with the **real difference in repayment**. If the model's gap is much bigger than the real difference, the strict 5/6 threshold is magnifying it. If it is similar, the gap mostly reflects the data, which still raises the question of whether the data itself is fair (section 6.2).

In [ ]:
real_gap = yv[~young].mean() - yv[young].mean()
print(f"Real difference in good-risk rate (25+ minus under 25): {real_gap * 100:+.1f} points")
print(f"Model's approval gap:                                   {point['approval gap'] * 100:+.1f} points")
print(f"Amplification: the model's gap is {point['approval gap'] / real_gap:.1f}x the real difference")

### 2.2 Fairness for all three models
The model comparison from section 1, judged on all three things: **accuracy, cost and fairness**. Is the most accurate model also the fairest?

In [ ]:
compare = {}
for name, p in oof.items():
    a = p > THRESHOLD
    _, pt, c = fairness_summary(a)
    compare[name] = {"accuracy (50% threshold)": results.loc[name, "accuracy at 50% threshold"],
                     "accuracy": results.loc[name, "accuracy"], "cost": results.loc[name, "cost"],
                     "approval gap (points)": pt["approval gap"] * 100,
                     "gap 95% CI": f"{c['approval gap'][0] * 100:+.0f} to {c['approval gap'][1] * 100:+.0f}",
                     "good-risk approval gap (points)": pt["good-risk approval gap"] * 100,
                     "approval ratio": pt["approval ratio (young / older)"]}
pd.DataFrame(compare).T.style.format({"accuracy (50% threshold)": "{:.1%}", "accuracy": "{:.1%}", "cost": "{:,.0f}", "approval gap (points)": "{:+.1f}",
                                      "good-risk approval gap (points)": "{:+.1f}", "approval ratio": "{:.2f}"})

### 2.3 Does a stricter bank create a bigger gap?
The threshold is a business choice. This chart shows the approval rate for each age group, and the gap between them, as the bank becomes stricter.

In [ ]:
ts = np.linspace(0.3, 0.95, 66)
yr = [(p_best[young] > t).mean() for t in ts]
orr = [(p_best[~young] > t).mean() for t in ts]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.fill_between(ts, yr, orr, color=GREY, alpha=0.18, lw=0, label="Gap")
ax.plot(ts, orr, color=BLUE, lw=2, label="25 or older")
ax.plot(ts, yr, color=ORANGE, lw=2, label="Under 25")
ax.axvline(0.5, color="#52514e", lw=1, ls=":"); ax.text(0.505, 0.04, "50%: accuracy-focused", fontsize=9)
ax.axvline(THRESHOLD, color="#52514e", lw=1, ls="--"); ax.text(THRESHOLD + 0.005, 0.92, "5/6: cost-focused", fontsize=9)
ax.set_xlabel("Approve if the model is at least this sure"); ax.set_ylabel("Share of group approved")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
r50 = (p_best[young] > 0.5).mean() / (p_best[~young] > 0.5).mean()
r56 = (p_best[young] > THRESHOLD).mean() / (p_best[~young] > THRESHOLD).mean()
ax.set_title(f"Stricter lending: young applicants fall from {r50:.0%} to {r56:.0%} of older applicants' approval rate", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower left")
plt.tight_layout(); plt.show()

for t, label in [(0.5, "50% threshold"), (THRESHOLD, "5/6 threshold")]:
    a_y, a_o = (p_best[young] > t).mean(), (p_best[~young] > t).mean()
    print(f"{label}: under 25 {a_y:.1%}, 25+ {a_o:.1%}, gap {100 * (a_o - a_y):.1f} points, ratio {a_y / a_o:.2f}")

### 2.4 Does the result depend on choosing 25?
Many papers split German Credit at **25 or younger** vs older (190 people), while we use **under 25** (149). The cut-off is a convention, not a natural boundary, so we re-run the audit at every cut-off from 22 to 35. If the gap only appeared at one cut-off, our finding would be fragile.

In [ ]:
ages_all = X["age"].to_numpy()
cut_rows = []
for c in range(22, 36):
    g = ages_all < c
    a = p_best > THRESHOLD
    cut_rows.append({"younger group": f"under {c}" + ("  (= 25 or younger)" if c == 26 else "  (ours)" if c == 25 else ""),
                     "people in group": int(g.sum()),
                     "approved, younger": a[g].mean(), "approved, older": a[~g].mean(),
                     "approval gap (points)": 100 * (a[~g].mean() - a[g].mean()),
                     "good-risk approval gap (points)": 100 * (a[~g & (yv == 1)].mean() - a[g & (yv == 1)].mean()),
                     "ratio": a[g].mean() / a[~g].mean()})
cuts = pd.DataFrame(cut_rows).set_index("younger group")
display(cuts.style.format({"approved, younger": "{:.1%}", "approved, older": "{:.1%}", "approval gap (points)": "{:.1f}",
                           "good-risk approval gap (points)": "{:.1f}", "ratio": "{:.2f}"}))
print(f"Approval gap ranges from {cuts['approval gap (points)'].min():.1f} to {cuts['approval gap (points)'].max():.1f} points; "
      f"ratio from {cuts['ratio'].min():.2f} to {cuts['ratio'].max():.2f}.")

### 2.5 The numbers the brief asks for
Reported at both operating points, because accuracy and cost disagree about which threshold is best.

In [ ]:
print(f"REPORT: {BEST}\n")
for t, label in [(0.5, "Accuracy-focused (50% threshold)"), (THRESHOLD, "Cost-focused (5/6 threshold, from the 5:1 rule)")]:
    a = p_best > t
    ev = evaluate(y, a)
    gm_t, pt_t, ci_t = fairness_summary(a)
    print(label)
    print(f"  Accuracy:                    {ev['accuracy']:.1%}   (approve-everyone: 70.0%)")
    print(f"  Cost:                        {ev['cost']:,}   (approve-everyone: 1,500; reject-everyone: 700)")
    print(f"  Approval rate, under 25:     {gm_t.loc['Under 25', 'approval rate']:.1%}")
    print(f"  Approval rate, 25 or older:  {gm_t.loc['25 or older', 'approval rate']:.1%}")
    print(f"  Gap:                         {pt_t['approval gap'] * 100:.1f} percentage points "
          f"(95% CI {ci_t['approval gap'][0] * 100:.1f} to {ci_t['approval gap'][1] * 100:.1f})")
    print(f"  Ratio (young / older):       {pt_t['approval ratio (young / older)']:.2f}   (below 0.80 = red flag)")
    print(f"  MAIN MEASURE - approved among good risks: under 25 {gm_t.loc['Under 25', 'approval rate among good risks']:.1%}, "
          f"25+ {gm_t.loc['25 or older', 'approval rate among good risks']:.1%}, gap {pt_t['good-risk approval gap'] * 100:.1f} points "
          f"(95% CI {ci_t['good-risk approval gap'][0] * 100:.1f} to {ci_t['good-risk approval gap'][1] * 100:.1f})\n")

### 2.6 What if bad loans were as rare as in real life?
The data has 30% bad loans, but the real rate is reported to be about 5%. The model learned from the 30% version, so it is too pessimistic about everyone. Correcting its odds to a 5% base rate, the same 5:1 rule ("lend above 5/6") becomes a much lower cut-off on the model's scores. So the 0.50 ratio is a result **for this oversampled data**: a real bank would be less strict, and the gap smaller.

In [ ]:
REAL_BAD = 0.05
data_bad = 1 - yv.mean()
# odds of 'good' scale by (real good:bad odds) / (data good:bad odds)
odds_factor = ((1 - REAL_BAD) / REAL_BAD) / ((1 - data_bad) / data_bad)
odds_needed = (THRESHOLD / (1 - THRESHOLD)) / odds_factor
T_REAL = odds_needed / (1 + odds_needed)
a_real = p_best > T_REAL
gm_r, pt_r, ci_r = fairness_summary(a_real)
print(f"Equivalent cut-off on the model's scores at a {REAL_BAD:.0%} bad rate: {T_REAL:.2f} (vs {THRESHOLD:.2f})")
print(f"Approved: under 25 {gm_r.loc['Under 25', 'approval rate']:.1%}, 25+ {gm_r.loc['25 or older', 'approval rate']:.1%}")
print(f"Gap {pt_r['approval gap'] * 100:.1f} points, ratio {pt_r['approval ratio (young / older)']:.2f}, "
      f"good-risk gap {pt_r['good-risk approval gap'] * 100:.1f} points")

## 3. What drives the gap?

We explain **logistic regression**: our chosen model, and the one whose reasoning we can read directly.

Logistic regression gives every applicant a **score**: a weighted sum of their details. Higher score means more likely to be approved. Because the score is just a sum, the difference in *average* score between older and younger applicants splits exactly into one piece per detail:

$$\text{gap from a detail} = \text{weight} \times \text{(how much the groups differ on it)}$$

So a detail only drives the gap if the model cares about it (large weight) **and** young and older applicants differ on it.

In [ ]:
# Fit logistic regression on all 1,000 applicants (settings tuned by cross-validation) so we can read its weights
lr_pipe, lr_grid = make_models()["Logistic regression"]
lr_full = GridSearchCV(lr_pipe, lr_grid, cv=INNER, scoring="neg_log_loss").fit(X, y).best_estimator_
prep, lr = lr_full.named_steps["prep"], lr_full.named_steps["model"]
Z = prep.transform(X)
col_names = prep.get_feature_names_out()

def original_feature(name):
    kind, rest = name.split("__", 1)
    if kind == "num":
        return rest
    return max((c for c in CAT if rest.startswith(c + "_")), key=len)

contrib = pd.Series(lr.coef_[0] * (Z[~young].mean(0) - Z[young].mean(0)), index=col_names)
by_feature = contrib.groupby([original_feature(n) for n in col_names]).sum().sort_values()
score = Z @ lr.coef_[0]
print(f"Average score gap (25+ minus under 25): {score[~young].mean() - score[young].mean():.3f}")
print(f"Sum of the per-detail pieces:            {by_feature.sum():.3f}   (they match exactly)")

In [ ]:
top = by_feature.reindex(by_feature.abs().sort_values().index)[-10:]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(top.index.str.replace("_", " "), top.values, color=[BLUE if v > 0 else ORANGE for v in top.values], height=0.6)
ax.axvline(0, color="#52514e", lw=1)
ax.set_xlim(min(top.min() * 1.6, -0.05), top.max() * 1.15)
for i, v in enumerate(top.values):
    ax.text(v + (0.004 if v > 0 else -0.004), i, f"{v:+.2f}", va="center", ha="left" if v > 0 else "right", fontsize=9)
ax.set_xlabel("Contribution to the score gap  (blue: favours older applicants · orange: favours young)")
ax.set_title("What pushes young applicants' scores down", loc="left", fontsize=11)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**How do the groups actually differ on these details?** For each of the top drivers, the category where young and older applicants differ most:

In [ ]:
rows = []
for f in by_feature.abs().sort_values(ascending=False).index[:6]:
    if f in NUM:
        rows.append({"detail": f, "value": "average", "under 25": X.loc[young, f].mean(), "25 or older": X.loc[~young, f].mean()})
    else:
        share_y = X.loc[young, f].value_counts(normalize=True)
        share_o = X.loc[~young, f].value_counts(normalize=True)
        diff = (share_y.reindex(share_o.index, fill_value=0) - share_o).abs()
        cat = diff.idxmax()
        rows.append({"detail": f, "value": cat, "under 25": f"{share_y.get(cat, 0):.0%}", "25 or older": f"{share_o[cat]:.0%}"})
pd.DataFrame(rows).set_index("detail")

### 3.1 Is the gap justified?
A useful way to judge each driver is to ask **"does this detail measure whether someone can repay, or just how far along in life they are?"**

- **Repayment-related** (stronger case for using them): checking account balance, credit history, loan size and length.
- **Life-stage related** (weaker case): age itself, years in current job, years at current address, renting vs owning. Young people score badly on these almost by definition, even if they are reliable.

If a large share of the gap comes from life-stage details, the model is partly penalising people **for being young**, not for being risky. Section 4 tests this directly.

## 4. Proxy test: if we delete age, does the bias go away?

The obvious fix is "just don't let the model see age". This is called **fairness through unawareness**. We test whether it works.

In [ ]:
NUM_NOAGE = [c for c in NUM if c != "age"]
X_noage = X.drop(columns="age")

def fairness_row(p, label, t=THRESHOLD):
    a = p > t
    ev = evaluate(y, a)
    gm_, pt_, ci_ = fairness_summary(a)
    return {"option": label, "accuracy": ev["accuracy"], "cost": ev["cost"],
            "approved, under 25": gm_.loc["Under 25", "approval rate"],
            "approved, 25+": gm_.loc["25 or older", "approval rate"],
            "gap (points)": pt_["approval gap"] * 100,
            "gap 95% CI": f"{ci_['approval gap'][0] * 100:.0f} to {ci_['approval gap'][1] * 100:.0f}",
            "ratio": pt_["approval ratio (young / older)"],
            "good-risk gap (points)": pt_["good-risk approval gap"] * 100}

FMT = {"good-risk gap (points)": "{:.1f}", "accuracy": "{:.1%}", "cost": "{:,.0f}", "approved, under 25": "{:.1%}", "approved, 25+": "{:.1%}",
       "gap (points)": "{:.1f}", "ratio": "{:.2f}"}

pipe, grid = make_models(CAT, NUM_NOAGE)["Logistic regression"]
p_noage, _, _ = out_of_fold_probs(pipe, grid, X_noage, y)
options = [fairness_row(oof["Logistic regression"], "Original (sees age)"),
           fairness_row(p_noage, "Age deleted")]
pd.DataFrame(options).set_index("option").style.format(FMT)

### 4.1 Which details give age away?
If the other details can **predict** whether someone is under 25, the model can still "see" age indirectly. These details are called **proxies**.

We train a model to predict *"is this person under 25?"* using everything except age. Then we test each detail on its own.

**How to read AUC:** 0.5 means no better than guessing, 1.0 means perfect.

In [ ]:
from sklearn.model_selection import cross_val_predict
y_young = pd.Series(young.astype(int), name="under_25")

pipe, grid = make_models(CAT, NUM_NOAGE)["Logistic regression"]
p_young, _, _ = out_of_fold_probs(pipe, grid, X_noage, y_young)
print(f"All details together predict 'under 25' with AUC {roc_auc_score(y_young, p_young):.2f}\n")

single = {}
for f in X_noage.columns:
    cat_f, num_f = ([f], []) if f in CAT else ([], [f])
    one = Pipeline([("prep", make_preprocessor(cat_f, num_f)), ("model", LogisticRegression(max_iter=5000))])
    single[f] = roc_auc_score(y_young, cross_val_predict(one, X_noage[[f]], y_young, cv=OUTER, method="predict_proba")[:, 1])
proxy_auc = pd.Series(single).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
top_p = proxy_auc[-10:]
ax.barh(top_p.index.str.replace("_", " "), top_p.values - 0.5, left=0.5, color=BLUE, height=0.6)
for i, v in enumerate(top_p.values):
    ax.text(v + 0.003, i, f"{v:.2f}", va="center", fontsize=9)
ax.axvline(0.5, color="#52514e", lw=1); ax.text(0.502, len(top_p) - 0.4, "0.5 = guessing", fontsize=8, color="#52514e")
ax.set_xlim(0.48, max(0.8, top_p.max() + 0.05))
ax.set_xlabel("AUC for predicting 'under 25' from this detail alone")
ax.set_title("Details that give away a person's age", loc="left", fontsize=11)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
TOP_PROXIES = proxy_auc.index[::-1][:3].tolist()
print("Top 3 proxies:", TOP_PROXIES)

### 4.2 What if we also delete the proxies?

In [ ]:
X_noproxy = X_noage.drop(columns=TOP_PROXIES)
cat_np = [c for c in CAT if c not in TOP_PROXIES]
num_np = [c for c in NUM_NOAGE if c not in TOP_PROXIES]
pipe, grid = make_models(cat_np, num_np)["Logistic regression"]
p_noproxy, _, _ = out_of_fold_probs(pipe, grid, X_noproxy, y)
options.append(fairness_row(p_noproxy, f"Age + top 3 proxies deleted"))
pd.DataFrame(options).set_index("option").style.format(FMT)

## 5. Fixing the gap, and what it costs

We try three different kinds of fix. All are judged on the same two things: **the gap** and **the cost** (5:1 rule).

| Fix | How it works | Uses age when deciding? |
|---|---|---|
| **Delete age / proxies** (section 4) | Hide the information | No |
| **Reweighing** | During training, give extra weight to under-represented combinations (young + good risk, older + bad risk) so "young = risky" is harder to learn as a shortcut | No |
| **Separate thresholds** | Keep the model, but let under-25s be approved at a slightly lower confidence | **Yes** |

The last fix is the most direct, but deciding differently *because of* someone's age can itself be unlawful in many countries. That tension is worth raising in the presentation.

In [ ]:
# Reweighing (Kamiran & Calders): weight = P(group) x P(label) / P(group, label)
g = young.astype(int); yv = y.to_numpy()
w = np.ones(len(y))
for gi in (0, 1):
    for yi in (0, 1):
        m = (g == gi) & (yv == yi)
        w[m] = (g == gi).mean() * (yv == yi).mean() / m.mean()
print(pd.crosstab(np.where(young, "Under 25", "25 or older"), np.where(yv == 1, "good", "bad"),
                  values=w, aggfunc="first").round(2).rename_axis(index="weight given to", columns=None))

pipe, grid = make_models(CAT, NUM_NOAGE)["Logistic regression"]
p_reweigh, _, _ = out_of_fold_probs(pipe, grid, X_noage, y, sample_weight=w)
options.append(fairness_row(p_reweigh, "Age deleted + reweighing"))

In [ ]:
# Separate thresholds: older applicants keep 5/6; try every lower threshold for under-25s
p_lr = oof["Logistic regression"]
sweep = []
for t_y in np.linspace(0.40, THRESHOLD, 90):
    a = np.where(young, p_lr > t_y, p_lr > THRESHOLD)
    r_y, r_o = a[young].mean(), a[~young].mean()
    tpr_y, tpr_o = a[young & (yv == 1)].mean(), a[~young & (yv == 1)].mean()
    sweep.append({"t_young": t_y, "cost": evaluate(y, a)["cost"], "gap": (r_o - r_y) * 100,
                  "ratio": r_y / r_o, "tpr_gap": (tpr_o - tpr_y) * 100})
sweep = pd.DataFrame(sweep)

def threshold_option(rows, label):
    t_y = rows.sort_values("t_young").iloc[-1]["t_young"]    # strictest (cheapest) threshold that meets the target
    a = np.where(young, p_lr > t_y, p_lr > THRESHOLD)
    row = fairness_row(a.astype(float), f"{label} (under-25 threshold {t_y:.2f})", t=0.5)
    return row

options.append(threshold_option(sweep[sweep["ratio"] >= 0.8], "Separate threshold: ratio 0.8"))
options.append(threshold_option(sweep[sweep["tpr_gap"] <= 0], "Separate threshold: equal chance for good risks"))
options.append(threshold_option(sweep[sweep["gap"] <= 0], "Separate threshold: equal approval rates"))
# The same fix with the under-25 cut-off chosen OUT OF FOLD: for each test fold, the cut-off is picked
# using only the other four folds, so the test predictions never influence it.
def pick_t_young(mask):
    best = 0.40
    for t_y in np.linspace(0.40, THRESHOLD, 90):
        a = np.where(young, p_lr > t_y, p_lr > THRESHOLD)[mask]
        g_ = young[mask]
        if a[g_].mean() / a[~g_].mean() >= 0.8:
            best = t_y                                   # keep the strictest cut-off that still reaches 0.8
    return best
a_fix_oof = np.zeros(len(yv), bool)
T_OOF = []
for k in range(5):
    t_k = pick_t_young(FOLDS != k); T_OOF.append(round(t_k, 2))
    te = FOLDS == k
    a_fix_oof[te] = np.where(young[te], p_lr[te] > t_k, p_lr[te] > THRESHOLD)
options.append(fairness_row(a_fix_oof.astype(float), "Separate threshold: ratio 0.8, chosen out of fold", t=0.5))
print("Under-25 cut-offs chosen out of fold, one per test fold:", T_OOF)
fixes = pd.DataFrame(options).set_index("option")
base_cost = fixes.loc["Original (sees age)", "cost"]
fixes["extra cost vs original"] = (fixes["cost"] / base_cost - 1)
fixes.style.format(FMT | {"extra cost vs original": "{:+.1%}"})

**Chart: the price of fairness.** Each dot is one option. Further left = fairer, lower = cheaper. The line shows every possible separate threshold for under-25s.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
near = sweep[sweep["gap"] >= -6]
ax.plot(near["gap"], near["cost"], color=GREY, lw=1.5, zorder=1, label="Separate threshold for under-25s (every setting)")
styles = {"Original": ("#0b0b0b", "o"), "Age deleted + reweighing": ("#1baf7a", "^"),
          "Age deleted": (ORANGE, "s"), "Age +": (ORANGE, "D"), "Separate": (BLUE, "o")}
for n, (opt, r) in enumerate(fixes.iterrows(), start=1):
    colour, mk = next(v for k, v in styles.items() if opt.startswith(k))
    ax.scatter(r["gap (points)"], r["cost"], s=80, color=colour, marker=mk, zorder=3, edgecolor="white", linewidth=1.5,
               label=f"{n}. {opt.split(' (')[0]}  (cost {r['cost']:,.0f})")
    ax.annotate(str(n), (r["gap (points)"], r["cost"]), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=9, weight="bold")
ax.set_xlabel("Approval gap: 25+ minus under 25 (percentage points)   ← fairer")
ax.set_ylabel("Cost (5:1 rule)   ← cheaper")
ax.set_title("Closing the age gap: no measurable cost to the bank", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8.5)
plt.tight_layout(); plt.show()

In [ ]:
o = fixes.loc["Original (sees age)"]
print("PRICE OF FAIRNESS (compared with the original model)\n")
for opt, r in fixes.iterrows():
    if opt == "Original (sees age)":
        continue
    print(f"{opt}")
    print(f"   gap {o['gap (points)']:.1f} -> {r['gap (points)']:.1f} points, "
          f"ratio {o['ratio']:.2f} -> {r['ratio']:.2f}, cost {o['cost']:,.0f} -> {r['cost']:,.0f} ({r['extra cost vs original']:+.1%})")

**How to read this:** the honest comparison is the out-of-fold version of the separate threshold, because its cut-off never saw the test data. The next cell checks whether its cost differs from the original by more than chance, and who the extra approvals actually are.

**Caveat for honesty:** the three separate-threshold rows above it chose their cut-off on the same out-of-sample predictions we then score them on, so their cost (541 to 554) is slightly optimistic. The out-of-fold row removes that problem and is the one we quote. With more time we would go one step further and choose the cut-off with a full inner cross-validation inside each training fold.

In [ ]:
# Paired bootstrap: resample applicants, compare the fix and the original on the SAME resample each time
a_orig = p_lr > THRESHOLD
boot = np.random.default_rng(SEED).integers(0, len(yv), size=(2000, len(yv)))
def boot_cost(a):
    A, Y = a[boot], yv[boot]
    return 5 * ((Y == 0) & A).sum(1) + ((Y == 1) & ~A).sum(1)
diff = boot_cost(a_fix_oof) - boot_cost(a_orig)
lo, hi = np.percentile(diff, [2.5, 97.5])
print(f"Cost: original {evaluate(y, a_orig)['cost']}, fix (out of fold) {evaluate(y, a_fix_oof)['cost']}")
print(f"Cost difference (fix minus original), 95% interval: {lo:+.0f} to {hi:+.0f}  ->  no measurable change in cost" if lo < 0 < hi
      else f"Cost difference 95% interval: {lo:+.0f} to {hi:+.0f}")

extra = a_fix_oof & ~a_orig & young
print(f"\nWho the fix adds: {extra.sum()} extra under-25 approvals, of whom {int(yv[extra].sum())} actually repaid "
      f"and {int((1 - yv[extra]).sum())} did not.")
gm_f, pt_f, ci_f = fairness_summary(a_fix_oof)
print(f"Main measure (approved among good risks): gap {point['good-risk approval gap'] * 100:.1f} -> {pt_f['good-risk approval gap'] * 100:.1f} points")

## 6. Flip test, and an audit of the data itself

### 6.1 Flip test: change only the age
Take each applicant, change **nothing except their age**, and ask the model again. If decisions change, age itself is driving them.

We use the logistic regression trained on all 1,000 applicants (section 3).

In [ ]:
def p_good(df):
    return lr_full.predict_proba(df[COLS])[:, 1]

p_now = p_good(X)
X_35 = X.copy(); X_35.loc[young, "age"] = 35          # every under-25 becomes 35
X_22 = X.copy(); X_22.loc[~young, "age"] = 22         # every 25+ becomes 22
p_as35, p_as22 = p_good(X_35), p_good(X_22)

app_now = p_now > THRESHOLD
gained = (~app_now & (p_as35 > THRESHOLD))[young]
lost = (app_now & (p_as22 <= THRESHOLD))[~young]
print(f"Under-25s rejected now but approved if they were 35: {gained.sum()} of {young.sum()} "
      f"({gained.mean():.0%})")
print(f"25+ approved now but rejected if they were 22:       {lost.sum()} of {(~young).sum()} "
      f"({lost.mean():.0%})")
print(f"\nAverage change in approval chance for under-25s made 35: {100 * (p_as35 - p_now)[young].mean():+.1f} points")
flipped_young = np.where(young & ~app_now & (p_as35 > THRESHOLD))[0]
EX_IDX = flipped_young[0] if len(flipped_young) else int(np.where(young)[0][np.argmax(p_now[young])])

**Chart: approval chance by age for one real young applicant** (one whose decision flips). All their details stay fixed; only age changes. The fairer model from section 5 (age deleted, reweighed) is shown for comparison.

In [ ]:
fair_pipe, fair_grid = make_models(CAT, NUM_NOAGE)["Logistic regression"]
fair_full = GridSearchCV(fair_pipe, fair_grid, cv=INNER, scoring="neg_log_loss").fit(
    X_noage, y, model__sample_weight=w).best_estimator_

typical = {c: X[c].mode()[0] for c in CAT} | {c: int(X[c].median()) for c in NUM}
ages = np.arange(19, 71)
person = X.iloc[EX_IDX].to_dict()
curve = pd.DataFrame([person | {"age": a} for a in ages])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ages, p_good(curve), color=BLUE, lw=2, label="Original model (sees age)")
ax.plot(ages, fair_full.predict_proba(curve.drop(columns="age"))[:, 1], color="#1baf7a", lw=2, ls="--",
        label="Fairer model (age deleted + reweighing)")
ax.axhline(THRESHOLD, color="#52514e", lw=1, ls=":"); ax.text(70, THRESHOLD + 0.002, "approval line (5/6)", ha="right", va="bottom", fontsize=9)
ax.axvspan(19, 25, color=ORANGE, alpha=0.08, lw=0)
ax.text(22, ax.get_ylim()[1], "under 25", color=ORANGE, fontsize=9, ha="center", va="top")
ax.set_xlabel("Age (everything else identical)"); ax.set_ylabel("Model's chance of 'good risk'")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"Applicant #{EX_IDX}: same finances, but only approved once they are older", loc="left", fontsize=11)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
cross = ages[p_good(curve) > THRESHOLD]
print(f"Applicant #{EX_IDX} is really {person['age']}; the original model would first approve them at age {cross.min() if len(cross) else 'never'}")

**Important limitation of the flip test:** it only catches age's *direct* effect. Section 3 showed most of the gap comes *through* details linked to age (renting, job length and so on). A model with age deleted would pass the flip test perfectly and still have a large gap (section 4). So passing a flip test does **not** prove a model is fair.

One more thing to notice: logistic regression treats age as a straight line, so it assumes "older is always safer". In reality, risk may rise again for the very old. A curved age effect would be a good extension.

### 6.2 Questioning the data
Before trusting any fairness result, we checked the data itself. Several problems are documented in a corrected version of the dataset (*South German Credit*, Grömping 2019, on the UCI repository):

In [ ]:
audit = pd.DataFrame([
    {"issue": "Sex cannot be recovered",
     "evidence in our data": "status codes: " + ", ".join(f"{k.split(' (')[0]} = {v}" for k, v in X['personal_status_sex'].value_counts().items()),
     "why it matters": "Code 2 mixes married women and single men, so any 'women' analysis is meaningless. Many published fairness studies used it anyway."},
    {"issue": "Bad loans heavily oversampled",
     "evidence in our data": f"{(1 - yv.mean()):.0%} bad in the data",
     "why it matters": "The real share of bad loans is reported to be only about 5% (Grömping 2019, relaying the original German sources), so costs and default chances are inflated, and approval rates are too low, compared with a real bank (section 2.6)."},
    {"issue": "Old data",
     "evidence in our data": "loans from 1973–75",
     "why it matters": "Lending, jobs and housing for young people have changed a lot since then."},
    {"issue": "Only people who were given loans",
     "evidence in our data": "all 1,000 rows are granted credits",
     "why it matters": "Applicants the bank rejected back then never appear, so we can't see whether they would have repaid. Any past bias in who the bank approved is baked into the data."},
    {"issue": "Some categories behave strangely",
     "evidence in our data": "good-risk rate: 'critical account' " + f"{yv[X['credit_history'] == 'critical account'].mean():.0%}"
                             + ", everyone else " + f"{yv[X['credit_history'] != 'critical account'].mean():.0%}",
     "why it matters": "A 'critical' credit history looking *safer* suggests more category names are wrong, so explanations of which details matter should be treated with caution."},
    {"issue": "Foreign worker coding looks reversed",
     "evidence in our data": "counts: " + ", ".join(f"{k} = {v}" for k, v in X['foreign_worker'].value_counts().items()),
     "why it matters": "Taken literally, over 95% of borrowers would be foreign workers, which is implausible for a 1970s German bank."},
]).set_index("issue")
audit.style.set_properties(**{"text-align": "left", "white-space": "normal"})

### 6.3 Deeper fairness checks
Three extra checks on our main result. Each answers a question a sceptical judge would ask:

| Check | The question in plain English |
|---|---|
| **6.3.1 Calibration by age** | When the model says "83% sure", is that equally true for young and older applicants? |
| **6.3.2 Like for like** | If a young and an older applicant have similar finances, is there still a gap? |
| **6.3.3 Cliff edge** | Our fix draws a line at age 25. Is it fair to the 25-year-old just on the other side? |

Same rules as the rest of the notebook: every prediction is **out-of-fold** (made while that applicant was hidden from training), every interval comes from **2,000 bootstrap resamples** with a fixed seed, and nothing here is tuned on the test predictions. None of the earlier results change.

#### 6.3.1 Calibration by age: is "83% sure" equally true for both groups?
A model is **calibrated** if its confidence can be taken at face value: of the people it calls "80% likely to repay", about 80% really do. If the model were *underrating* young people, reliable young applicants would repay more often than it predicted. That would be a hidden unfairness inside the scores themselves.

We sort applicants into five confidence bands (set in advance; the top band is everyone above the 5/6 approval line) and compare what the model predicted with what actually happened, separately for each age group.

**How to read the chart:** dots on the dotted diagonal mean an honest model. Dots *above* it mean that group repaid more often than predicted (underrated); dots *below* mean overrated. The bars show the 95% interval, which is wide for under-25s because each band holds only 20 to 50 of them.

In [ ]:
# Calibration: when the model says "x% sure", how often did those people actually repay? Split by age group.
EDGES = [0, 0.50, 0.65, 0.75, THRESHOLD, 1.0]      # fixed in advance; 5/6 is an edge, so the last bin = approved
BIN_NAMES = ["under 50%", "50–65%", "65–75%", "75–83% (just below the line)", "above 83% (approved)"]
score_bin = np.digitize(p_lr, EDGES[1:-1], right=True)                              # which bin each applicant's score falls in
BOOT63 = np.random.default_rng(SEED).integers(0, len(yv), size=(2000, len(yv)))     # 2,000 resamples of the applicants
surprise = yv - p_lr                                 # +1 side: repaid more often than predicted (underrated)

def boot_mean(values, mask):
    """Average of `values` among the applicants in `mask`, with a 95% bootstrap interval."""
    M = mask[BOOT63]
    with np.errstate(invalid="ignore", divide="ignore"):
        b = (values[BOOT63] * M).sum(1) / M.sum(1)
    return values[mask].mean(), np.nanpercentile(b, 2.5), np.nanpercentile(b, 97.5)

def boot_diff(values, mask_a, mask_b):
    """Difference in averages (group a minus group b), with a 95% bootstrap interval."""
    A, Bm = mask_a[BOOT63], mask_b[BOOT63]
    with np.errstate(invalid="ignore", divide="ignore"):
        b = (values[BOOT63] * A).sum(1) / A.sum(1) - (values[BOOT63] * Bm).sum(1) / Bm.sum(1)
    return values[mask_a].mean() - values[mask_b].mean(), np.nanpercentile(b, 2.5), np.nanpercentile(b, 97.5)

AGE_GROUPS = [("Under 25", young, ORANGE), ("25 or older", ~young, BLUE)]
cal_rows, cal_points = [], {}
for label, grp, colour in AGE_GROUPS:
    pts = []
    for k, name in enumerate(BIN_NAMES):
        m63 = grp & (score_bin == k)
        act, a_lo, a_hi = boot_mean(yv.astype(float), m63)
        cal_rows.append({"group": label, "model's confidence": name, "people": int(m63.sum()),
                         "model predicted": p_lr[m63].mean(), "actually repaid": act,
                         "95% interval": f"{a_lo:.0%} to {a_hi:.0%}"})
        pts.append((p_lr[m63].mean(), act, a_lo, a_hi))
    cal_points[label] = pts
calibration = pd.DataFrame(cal_rows).set_index(["group", "model's confidence"])
display(calibration.style.format({"model predicted": "{:.1%}", "actually repaid": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot([0.25, 1], [0.25, 1], color=GREY, lw=1, ls=":", label="Perfectly honest model")
for (label, grp, colour), shift in zip(AGE_GROUPS, (-0.006, 0.006)):
    px, act, c_lo, c_hi = map(np.array, zip(*cal_points[label]))
    ax.errorbar(px + shift, act, yerr=[act - c_lo, c_hi - act], color=colour, marker="o", lw=2, capsize=3, label=label)
ax.axvline(THRESHOLD, color="#52514e", lw=1, ls="--"); ax.text(THRESHOLD + 0.008, 0.28, "approval line (5/6)", fontsize=9)
ax.set_xlabel("What the model predicted (average chance of repaying in each confidence band)")
ax.set_ylabel("Share who actually repaid")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("The model is about equally honest for both age groups", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

def reading(c_lo, c_hi):
    return ("UNDERRATED: repaid more often than predicted" if c_lo > 0 else
            "OVERRATED: repaid less often than predicted" if c_hi < 0 else "no measurable difference from the prediction")
print("Actual repayment minus the model's prediction, in percentage points (95% interval):\n")
for where, sel in [("Everyone", np.ones(len(yv), bool)), ("Rejected (below the 5/6 line)", score_bin < 4),
                   ("Just below the line (75–83%)", score_bin == 3)]:
    for label, grp, _ in AGE_GROUPS:
        d, d_lo, d_hi = boot_mean(surprise, grp & sel)
        print(f"  {where:30s} {label:12s} {100 * d:+5.1f}  ({100 * d_lo:+.1f} to {100 * d_hi:+.1f})   {reading(d_lo, d_hi)}")
    d, d_lo, d_hi = boot_diff(surprise, young & sel, ~young & sel)
    print(f"  {'':30s} {'young - older':12s} {100 * d:+5.1f}  ({100 * d_lo:+.1f} to {100 * d_hi:+.1f})\n")
band = score_bin == 3
print(f"Just below the line, {yv[young & band].mean():.0%} of under-25s ({int(yv[young & band].sum())} of {int((young & band).sum())}) and "
      f"{yv[~young & band].mean():.0%} of older applicants ({int(yv[~young & band].sum())} of {int((~young & band).sum())}) actually repaid. "
      f"The lending rule needs {THRESHOLD:.0%}.")

**What we found: no, the model does not underrate reliable young people.** Its confidence is about equally honest for both groups. Across all under-25s it predicted 61.3% would repay and 59.1% did (2.3 points fewer, 95% interval −9.7 to +4.9); for older applicants the difference is +0.5. No difference here is bigger than chance. The honest limit: with only 149 young applicants, a mismatch of up to about 10 points could hide in the noise.

This sharpens our story in two ways:
- The age gap is **not** caused by the model misjudging young people's risk. It comes from where the approval line falls: more young applicants sit just under it.
- Just under the line (scores of 75–83%), **both** groups did well: 85% of under-25s (17 of 20) and 83% of older applicants (107 of 129) repaid, which is right at the 83% the lending rule needs. These borderline applicants are about break-even for the bank whatever their age. That is why our fix costs nothing measurable, and it raises a question we answer in 6.3.3: what about the *older* borderline applicants?

#### 6.3.2 Like for like: is there still a gap between applicants with similar finances?
An obvious objection: "young people borrow differently and have less in the bank, so of course fewer are approved." This check compares like with like. It is the idea behind **conditional demographic disparity**, the measure recommended in the FCA's 2024 research note on bias: the gap between groups *after* allowing for legitimate differences.

We put every applicant into a group with the same checking-account status, the same savings band, and a similar loan amount and loan length (each split at the median). Inside each group we take the older approval rate minus the young approval rate, then average those gaps, giving more weight to groups with more young applicants. To check that our choices do not drive the answer, we repeat it with amount and length split into thirds.

**Honest caveat:** which details count as "legitimate" is a judgement, not a fact. We used the four details most clearly about the loan and the applicant's money, and left out life-stage details such as housing and job length (section 3.1).

In [ ]:
# Like-for-like: compare young and older applicants only within groups that share the same finances
def finance_groups(n_bins):
    """Label each applicant by checking account, savings, loan-amount band and loan-length band."""
    amount_band = pd.qcut(X["amount"], n_bins, labels=False, duplicates="drop").astype(str)
    length_band = pd.qcut(X["duration"], n_bins, labels=False, duplicates="drop").astype(str)
    label = X["status"].astype(str) + " | " + X["savings"].astype(str) + " | amount " + amount_band + " | length " + length_band
    return pd.factorize(label)[0]

def like_for_like(approve, codes, good_only=False, idx=None):
    """Raw gap, and the gap after matching on finances: within each finance group, older approval rate minus young
    approval rate, averaged with weights = number of young applicants in the group. Both in percentage points."""
    idx = np.arange(len(yv)) if idx is None else idx
    keep = (yv[idx] == 1) if good_only else np.ones(len(idx), bool)
    c, yg, a = codes[idx], young[idx] & keep, approve[idx].astype(float)
    og = ~young[idx] & keep
    n_groups = codes.max() + 1
    n_y, n_o = np.bincount(c, yg, n_groups), np.bincount(c, og, n_groups)
    a_y, a_o = np.bincount(c, yg * a, n_groups), np.bincount(c, og * a, n_groups)
    both = (n_y > 0) & (n_o > 0)                                  # finance groups that contain both ages
    matched = (n_y[both] * (a_o[both] / n_o[both] - a_y[both] / n_y[both])).sum() / n_y[both].sum()
    raw = a_o.sum() / n_o.sum() - a_y.sum() / n_y.sum()
    return 100 * raw, 100 * matched, n_y[both].sum() / n_y.sum()

lfl_rows = []
for n_bins, detail in [(2, "main: amount and length split at the median"), (3, "check: amount and length split in thirds")]:
    codes = finance_groups(n_bins)
    for policy, approve in [("Original model (5/6)", a_orig), ("Our fix (cut-off chosen out of fold)", a_fix_oof)]:
        for measure, good_only in [("all applicants", False), ("good risks only (main measure)", True)]:
            raw, matched, covered = like_for_like(approve, codes, good_only)
            bs = np.array([like_for_like(approve, codes, good_only, i)[:2] for i in BOOT63])
            m_lo, m_hi = np.percentile(bs[:, 1], [2.5, 97.5])
            lfl_rows.append({"matching": detail, "policy": policy, "gap measured among": measure,
                             "raw gap (points)": raw, "like-for-like gap (points)": matched,
                             "95% interval": f"{m_lo:+.1f} to {m_hi:+.1f}",
                             "share of raw gap left": matched / raw if policy.startswith("Original") else np.nan,
                             "young applicants matched": covered})
lfl = pd.DataFrame(lfl_rows).set_index(["matching", "policy", "gap measured among"])
display(lfl.style.format({"raw gap (points)": "{:+.1f}", "like-for-like gap (points)": "{:+.1f}",
                          "share of raw gap left": "{:.0%}", "young applicants matched": "{:.0%}"}, na_rep="–"))

main = lfl.loc["main: amount and length split at the median"]
o_all, o_good = main.loc[("Original model (5/6)", "all applicants")], main.loc[("Original model (5/6)", "good risks only (main measure)")]
f_all = main.loc[("Our fix (cut-off chosen out of fold)", "all applicants")]
print(f"{finance_groups(2).max() + 1} finance groups. Original model: approval gap {o_all['raw gap (points)']:.1f} points -> "
      f"{o_all['like-for-like gap (points)']:.1f} like for like ({o_all['95% interval']}), so {o_all['share of raw gap left']:.0%} of the gap "
      f"remains between applicants with similar finances.")
print(f"Good risks only: {o_good['raw gap (points)']:.1f} -> {o_good['like-for-like gap (points)']:.1f} points ({o_good['95% interval']}).")
print(f"After our fix, the like-for-like approval gap is {f_all['like-for-like gap (points)']:+.1f} points ({f_all['95% interval']}).")

**What we found: most of the gap survives.** Between applicants with similar finances, the approval gap is still **11.3 points** (95% interval 5.1 to 17.8), which is **59% of the raw 19.2-point gap**. So roughly 40% of the gap is explained by young people's different finances, and roughly 60% is not. On our main measure (good risks only), 13.6 of the 19.5 points remain (4.7 to 23.3).

The finer split gives a similar answer (10.0 points remain), but for good risks only its interval just touches zero (−0.5 to +19.0), because the groups become very small. So the safe statement is "about half or more of the gap remains like for like", not an exact share.

**After our fix, the like-for-like gap disappears:** −0.8 points (−8.2 to +6.6). Young and older applicants with similar finances are then approved at the same rate, within noise.

#### 6.3.3 Cliff edge: is the fix fair to the 25-year-old next door?
Our fix lowers the approval line for under-25s only. Any rule with an age line creates a cliff: a 24-year-old and a 26-year-old with the *same score* can now get different answers, this time in the young person's favour. A strict judge will ask about this, so we measure it.

- **Leapfrogged** applicants are people aged 25 or older who are still rejected, although their score would now get an under-25 approved.
- We also look at how often good risks are approved in each age band, before and after the fix, to see who is worst off afterwards.
- Finally we test the age-blind alternative: give **everyone** the lower line. It is the same out-of-fold line as in section 5 (chosen so young applicants reach a 0.8 ratio), so it was never tuned on cost or on these results.

In [ ]:
# Cliff-edge check: our fix draws a line at age 25. Who sits just on the wrong side of it?
t_young_oof = np.array([pick_t_young(FOLDS != k) for k in range(5)])[FOLDS]     # the out-of-fold under-25 cut-off that applied to each applicant
leapfrogged = ~young & (p_lr > t_young_oof) & (p_lr <= THRESHOLD)              # 25+, rejected, but with a score that gets an under-25 approved
extra_young = young & a_fix_oof & ~a_orig

AGE_BANDS = [("Under 25", 0, 25), ("25–29", 25, 30), ("30–39", 30, 40), ("40–59", 40, 60), ("60 or older", 60, 200)]
good_risk = yv == 1
band_rows, band_plot = [], []
for name, lo_age, hi_age in AGE_BANDS:
    in_band = (ages_all >= lo_age) & (ages_all < hi_age)
    before = boot_mean(a_orig.astype(float), in_band & good_risk)
    after = boot_mean(a_fix_oof.astype(float), in_band & good_risk)
    band_rows.append({"age band": name, "good risks in band": int((in_band & good_risk).sum()),
                      "approved before fix": before[0], "approved after fix": after[0],
                      "after: 95% interval": f"{after[1]:.0%} to {after[2]:.0%}",
                      "leapfrogged": int((leapfrogged & in_band).sum()),
                      "of whom repaid": int((leapfrogged & in_band & good_risk).sum())})
    band_plot.append((before, after))
bands_table = pd.DataFrame(band_rows).set_index("age band")
display(bands_table.style.format({"approved before fix": "{:.1%}", "approved after fix": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(8, 4.2))
bw = 0.36
for i, (before, after) in enumerate(band_plot):
    for j, (val, colour, lab) in enumerate([(before, GREY, "Before the fix"), (after, ORANGE if i == 0 else BLUE, None)]):
        xpos = i + (j - 0.5) * (bw + 0.02)
        ax.bar(xpos, val[0], width=bw, color=colour, label=lab if i == 0 else None)
        ax.errorbar(xpos, val[0], yerr=[[val[0] - val[1]], [val[2] - val[0]]], color="#0b0b0b", capsize=3, lw=1)
        ax.text(xpos, val[2] + 0.015, f"{val[0]:.0%}", ha="center", fontsize=9)
ax.bar(0, 0, color=ORANGE, label="After the fix (under 25)"); ax.bar(0, 0, color=BLUE, label="After the fix (25 or older: unchanged)")
ax.set_xticks(range(len(AGE_BANDS))); ax.set_xticklabels([b[0] for b in AGE_BANDS])
ax.set_ylim(0, 0.85); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylabel("Good risks approved")
ax.set_title("After the fix, 25–29-year-olds have the lowest approval rate among good risks", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()

print(f"The fix approves {int(extra_young.sum())} extra under-25s, of whom {int(yv[extra_young].sum())} repaid ({yv[extra_young].mean():.0%}).")
print(f"It leaves {int(leapfrogged.sum())} applicants aged 25+ rejected with scores that would get an under-25 approved; "
      f"{int(yv[leapfrogged].sum())} of them repaid ({yv[leapfrogged].mean():.0%}).")
d, d_lo, d_hi = boot_diff(a_fix_oof.astype(float), (ages_all >= 25) & (ages_all < 30) & good_risk, young & good_risk)
print(f"Good risks approved after the fix: 25–29 minus under 25 = {100 * d:+.1f} points (95% interval {100 * d_lo:+.1f} to {100 * d_hi:+.1f})"
      + ("  ->  within noise" if d_lo < 0 < d_hi else "") + ".")

# The age-blind alternative: give EVERYONE the lower cut-off (still the out-of-fold one, so nothing is tuned on test data)
a_lower_all = p_lr > t_young_oof
cliff_rows = []
for policy, approve in [("Original: 5/6 for everyone", a_orig), ("Our fix: lower cut-off for under-25s only", a_fix_oof),
                        ("Age-blind: the same lower cut-off for everyone", a_lower_all)]:
    gm_c, pt_c, ci_c = fairness_summary(approve)
    c_diff = boot_cost(approve) - boot_cost(a_orig)
    cliff_rows.append({"policy": policy, "cost": evaluate(y, approve)["cost"],
                       "cost vs original, 95% interval": "–" if approve is a_orig else f"{np.percentile(c_diff, 2.5):+.0f} to {np.percentile(c_diff, 97.5):+.0f}",
                       "approved": approve.mean(), "good-risk gap (points)": pt_c["good-risk approval gap"] * 100,
                       "gap 95% interval": f"{ci_c['good-risk approval gap'][0] * 100:+.0f} to {ci_c['good-risk approval gap'][1] * 100:+.0f}",
                       "ratio (young / older)": pt_c["approval ratio (young / older)"], "people leapfrogged": int(leapfrogged.sum()) if approve is a_fix_oof else 0})
cliff = pd.DataFrame(cliff_rows).set_index("policy")
display(cliff.style.format({"cost": "{:,.0f}", "approved": "{:.1%}", "good-risk gap (points)": "{:+.1f}", "ratio (young / older)": "{:.2f}"}))

**What we found: the fix has a side effect we should say out loud. It moves the cliff rather than removing it.**
- It approves 18 extra under-25s (15 repaid, 83%). But it leaves **122 applicants aged 25 or older** rejected with scores that would now get a young person approved, and 106 of them (87%) actually repaid.
- 33 of those are aged 25–29. After the fix, good risks aged 25–29 are approved 35% of the time, against 45% for under-25s: the lowest of any age band. That difference (−10.5 points, interval −23.2 to +3.2) is within noise, but the fix does nothing for a group that started almost as badly off (35% against 28%).
- The age-blind alternative shows the trade-off. Lowering the line for everyone costs no more (521 against 547; difference −73 to +24, so no measurable change) and leapfrogs nobody, but the age gap comes straight back: 19.7 points among good risks, ratio 0.59.

**What this means for our story.** On this data the bank can lend to more borderline applicants at no measurable cost. Closing the *age gap* means giving that benefit of the doubt to young applicants first. That is a choice about who goes first, and it should be made openly: it is not free for the older applicants who are leapfrogged. We think it is defensible, because young applicants are the ones held back by details linked to their age (section 3), but two softer versions are worth testing next (we have **not** tested them): phase the lower line out gradually between 25 and 30 instead of a hard cliff, or lower it for everyone and accept a smaller improvement (ratio 0.50 to 0.59).

## 7. Tail risk: what's the worst that could happen?

So far every loan counted the same. In reality:
- **Big loans lose more money** than small ones.
- **Defaults cluster.** In a recession many borrowers struggle at once.

### 7.1 Losses weighted by loan size
Approving a bad loan now costs **its full amount**; rejecting a good one costs **a fifth of its amount** (lost business), keeping the 5:1 ratio.

In [ ]:
amt = X["amount"].to_numpy().astype(float)
def money_cost(approve):
    approve = np.asarray(approve, bool)
    return amt[approve & (yv == 0)].sum() + amt[~approve & (yv == 1)].sum() / 5

T_08 = sweep.loc[sweep["ratio"] >= 0.8, "t_young"].max()
policies = {"Approve everyone": np.ones(len(yv), bool),
            "Reject everyone": np.zeros(len(yv), bool),
            "Original model (5/6)": p_lr > THRESHOLD,
            "Fair fix: separate threshold (ratio 0.8)": np.where(young, p_lr > T_08, p_lr > THRESHOLD),
            "Fair fix: age deleted + reweighing": p_reweigh > THRESHOLD}
money = pd.DataFrame({k: {"cost in DM": money_cost(a), "cost (count, 5:1)": evaluate(y, a)["cost"]} for k, a in policies.items()}).T
money["cost in DM, vs original"] = money["cost in DM"] / money.loc["Original model (5/6)", "cost in DM"] - 1
display(money.style.format({"cost in DM": "{:,.0f}", "cost (count, 5:1)": "{:,.0f}", "cost in DM, vs original": "{:+.0%}"}))
print(f"Average loan: under 25 {amt[young].mean():,.0f} DM, 25+ {amt[~young].mean():,.0f} DM")

### 7.2 Recession simulation and Value at Risk
We take the loans the model approves and simulate 20,000 possible futures. Each borrower defaults with the chance the model gives them.

- **Normal times:** each borrower's luck is independent.
- **With recessions:** every borrower also shares one "economy" dice. A bad roll pushes everyone towards default together (a *one-factor model*, the idea behind bank capital rules).
- **Stress test:** every default chance doubled.

**Value at Risk (VaR) 99%:** the loss that is only exceeded in the worst 1% of futures.

In [ ]:
from scipy.stats import norm
sim_rng = np.random.default_rng(SEED)
N_SIM, RHO = 20_000, 0.2             # RHO: how strongly borrowers share the economy's luck

def simulate(approve, pd_scale=1.0, rho=0.0):
    pd_i = np.clip((1 - p_lr[approve]) * pd_scale, 0, 0.999)
    exposure = amt[approve]
    economy = sim_rng.standard_normal((N_SIM, 1))
    own = sim_rng.standard_normal((N_SIM, len(pd_i)))
    latent = np.sqrt(rho) * economy + np.sqrt(1 - rho) * own
    return (latent < norm.ppf(pd_i)) @ exposure

approve_orig = p_lr > THRESHOLD
scenarios = {"Normal times (independent)": simulate(approve_orig),
             "With recessions (shared economy)": simulate(approve_orig, rho=RHO),
             "Stress: default chances doubled + recessions": simulate(approve_orig, pd_scale=2, rho=RHO)}
lent = amt[approve_orig].sum()
risk = pd.DataFrame({k: {"expected loss": v.mean(), "VaR 95%": np.percentile(v, 95), "VaR 99%": np.percentile(v, 99)}
                     for k, v in scenarios.items()}).T
risk["VaR 99% as % of money lent"] = risk["VaR 99%"] / lent
print(f"Approved {approve_orig.sum()} loans, {lent:,.0f} DM lent in total\n")
display(risk.style.format({"expected loss": "{:,.0f}", "VaR 95%": "{:,.0f}", "VaR 99%": "{:,.0f}", "VaR 99% as % of money lent": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, max(np.percentile(v, 99.9) for v in scenarios.values()), 80)
for (k, v), c in zip(scenarios.items(), [BLUE, ORANGE, "#e34948"]):
    ax.hist(v, bins=bins, histtype="step", lw=2, color=c, density=True, label=k)
    ax.axvline(np.percentile(v, 99), color=c, lw=1, ls=":")
ax.set_xlabel("Total loss on approved loans (DM)   ·   dotted lines: 99% VaR")
ax.set_yticks([]); ax.set_ylabel("How often")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f}k"))
ax.set_title("Recessions barely move the average loss but fatten the worst case", loc="left", fontsize=11)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

### 7.3 Does fairer lending make the bank riskier in a recession?
The key fairness question for tail risk. We run the same recession simulation for the fair fixes. To compare like with like, every policy's default chances come from the **same** risk estimate (the original model).

In [ ]:
fair_policies = {"Original model (5/6)": policies["Original model (5/6)"],
                 "Fair fix: separate threshold (ratio 0.8)": policies["Fair fix: separate threshold (ratio 0.8)"],
                 "Fair fix: age deleted + reweighing": policies["Fair fix: age deleted + reweighing"]}
rows = []
for name, a in fair_policies.items():
    sim_rng = np.random.default_rng(SEED)            # same random futures for every policy
    losses = simulate(a, rho=RHO)
    lent_a = amt[a].sum()
    young_share = amt[a & young].sum() / lent_a
    rows.append({"policy": name, "loans": int(a.sum()), "young borrowers": int((a & young).sum()),
                 "money lent (DM)": lent_a, "expected loss": losses.mean(), "VaR 99% (recessions)": np.percentile(losses, 99),
                 "VaR 99% as % lent": np.percentile(losses, 99) / lent_a, "share of lending to under-25s": young_share})
fair_risk = pd.DataFrame(rows).set_index("policy")
display(fair_risk.style.format({"money lent (DM)": "{:,.0f}", "expected loss": "{:,.0f}", "VaR 99% (recessions)": "{:,.0f}",
                                "VaR 99% as % lent": "{:.1%}", "share of lending to under-25s": "{:.1%}"}))
o, f = fair_risk.iloc[0], fair_risk.iloc[1]
print("Recession strength (RHO) is an assumption, so check how much it matters:")
for r_ in [0.05, 0.1, 0.2, 0.3]:
    vals = []
    for name, a in fair_policies.items():
        sim_rng = np.random.default_rng(SEED)
        vals.append(np.percentile(simulate(a, rho=r_), 99) / amt[a].sum())
    print(f"  RHO {r_:.2f}: worst-case loss original {vals[0]:.1%}, separate threshold {vals[1]:.1%}, reweighing {vals[2]:.1%}")
print(f"Separate threshold: {int(f['young borrowers'] - o['young borrowers'])} more young borrowers; "
      f"worst-case loss {o['VaR 99% as % lent']:.1%} -> {f['VaR 99% as % lent']:.1%} of money lent")

**Caveats:** the average loss is the same with or without recessions by construction (the shared economy changes how defaults cluster, not how likely each one is), so only the worst case is a finding. Default chances come from oversampled data (30% bad), so absolute losses are too high. The recession strength (`RHO = 0.2`) is an assumption, not something this dataset can measure. Try changing it and re-running.

## 8. Interactive demo
Enter an applicant's details. You'll see their approval chance, the main reasons, what changes if they were 35, and what the fairer model says.

*In Colab the controls are live. Below them, a real applicant from the data is shown as a worked example.*

In [ ]:
from IPython.display import HTML
from html import escape
nice = lambda s: s.replace("_", " ")

def explain(applicant):
    df = pd.DataFrame([applicant])[COLS]
    p = p_good(df)[0]
    p35 = p_good(df.assign(age=35))[0]
    p_fair = fair_full.predict_proba(df.drop(columns="age"))[0, 1]
    z = prep.transform(df)[0]
    reasons = pd.Series(lr.coef_[0] * (z - Z.mean(0)), index=col_names)
    reasons = reasons.groupby([original_feature(n) for n in col_names]).sum().sort_values()
    def describe(f):
        return f"{nice(f)}: <b>{escape(str(applicant[f]))}</b>"
    against = "".join(f"<li>{describe(f)}</li>" for f in reasons.index[:3] if reasons[f] < 0)
    for_ = "".join(f"<li>{describe(f)}</li>" for f in reasons.index[::-1][:3] if reasons[f] > 0)
    verdict = lambda q: ("<span style='color:#008300'><b>LEND</b></span>" if q > THRESHOLD
                         else "<span style='color:#e34948'><b>DON'T LEND</b></span>")
    flip = ""
    if applicant["age"] < 25:
        flip = (f"<p><b>Flip test:</b> the same person aged 35 → {p35:.1%} ({verdict(p35)})"
                + (" &nbsp;⚠️ <b>decision changes with age alone</b>" if (p > THRESHOLD) != (p35 > THRESHOLD) else "") + "</p>")
    return HTML(f'''
    <div style="font-family:sans-serif;border:1px solid #ddd;border-radius:8px;padding:12px 16px;max-width:620px">
      <p style="font-size:18px;margin:0">Chance of being a good risk: <b>{p:.1%}</b> → {verdict(p)}
         <span style="color:#777;font-size:13px">(lends above {THRESHOLD:.1%})</span></p>
      <p style="margin:8px 0 2px"><b>Main reasons against:</b></p><ul style="margin:0">{against or "<li>none</li>"}</ul>
      <p style="margin:8px 0 2px"><b>Main reasons for:</b></p><ul style="margin:0">{for_ or "<li>none</li>"}</ul>
      {flip}
      <p><b>Fairer model</b> (no age, reweighed): {p_fair:.1%} → {verdict(p_fair)}</p>
    </div>''')

# Worked example: a real under-25 applicant whose decision flips with age alone
ex = X.iloc[EX_IDX].to_dict()
print(f"Worked example: applicant #{EX_IDX}, age {ex['age']}, borrowing {ex['amount']:,} DM for {ex['duration']} months")
display(explain(ex))

In [ ]:
# Live controls (work in Colab / Jupyter)
import ipywidgets as wdg
SLIDERS = {"age": (18, 75, 1), "duration": (4, 72, 1), "amount": (250, 18500, 250), "installment_rate": (1, 4, 1),
           "present_residence": (1, 4, 1), "number_credits": (1, 4, 1), "people_liable": (1, 2, 1)}
controls = {}
for c in COLS:
    if c in SLIDERS:
        lo, hi, step = SLIDERS[c]
        controls[c] = wdg.IntSlider(value=int(typical[c]) if c != "age" else 22, min=lo, max=hi, step=step,
                                    description=nice(c), style={"description_width": "150px"}, layout=wdg.Layout(width="420px"))
    else:
        controls[c] = wdg.Dropdown(options=sorted(X[c].unique()), value=typical[c], description=nice(c),
                                   style={"description_width": "150px"}, layout=wdg.Layout(width="420px"))
out = wdg.interactive_output(lambda **a: display(explain(a)), controls)
display(wdg.HBox([wdg.VBox(list(controls.values())), out]))

### 8.1 The full walkthrough page
A step-by-step explanation of the whole project for someone with no background: the bank's problem, the two lazy baselines, how much our model saves, how we tested it, the unfairness, the fix, and a live demo. Run the cell below; nothing needs installing.

In [ ]:
#@title Show the interactive demo page { display-mode: "form" }
# The demo page is stored inside this cell so the notebook works on its own; it runs entirely in your browser.
import base64
from IPython.display import HTML, display
DEMO_PAGE = "..."
display(HTML('<iframe srcdoc="' + base64.b64decode(DEMO_PAGE).decode("utf-8").replace("&", "&amp;").replace('"', "&quot;") + '" style="width:100%;height:1400px;border:0;border-radius:12px;background:#000"></iframe>'))

## 9. Limitations, and how to overcome them

| Limitation | Why it matters | How to overcome it |
|---|---|---|
| **Small data:** 1,000 loans, 149 under 25 | Wide confidence intervals; results could shift with more data | Use a larger modern dataset (e.g. public US or Taiwan credit data) and repeat the audit |
| **Old data:** 1973–75, one German bank | Young people's finances and lending rules have changed | Re-run on recent data; compare with how UK lenders work today |
| **Oversampled bad loans** (30% bad vs a reported ~5% in reality) | Costs and default chances inflated; approval rates too low, so the gap may be smaller in reality (section 2.6) | Reweight to a realistic default rate before setting thresholds |
| **Only granted loans** | Rejected applicants are missing, so any past bias in who was approved is baked in | "Reject inference" methods, or data that includes rejected applicants |
| **Broken sex variable** | We can't test sex or intersectional bias | Use datasets that record sex reliably; push data owners to document coding |
| **Fix cut-off chosen out of fold, not in a full inner loop** (section 5) | The other folds' predictions come from models that saw the test fold, so a little optimism could remain | Choose the cut-off with an inner cross-validation inside each training fold |
| **No real recession in the data** | Tail-risk numbers rely on an assumed `RHO` | Calibrate to historical default rates across economic cycles |
| **One definition of fairness** | Equal approval rates and equal chances for good risks can't always both hold | Report several measures (we show three) and explain the trade-off |

In [ ]:
print("HEADLINE NUMBERS")
print(f"  Model: logistic regression, 5-fold cross-validated, threshold 5/6 from the 5:1 cost rule")
print(f"  Accuracy {results.loc['Logistic regression', 'accuracy at 50% threshold']:.1%} at 50% threshold "
      f"(approve-everyone 70.0%), cost {results.loc['Logistic regression', 'cost']:,.0f} at 5/6 (reject-everyone 700)")
print(f"  Approval: under 25 {gm.loc['Under 25', 'approval rate']:.1%}, 25+ {gm.loc['25 or older', 'approval rate']:.1%}, "
      f"gap {point['approval gap'] * 100:.1f} points (95% CI {ci['approval gap'][0] * 100:.0f} to {ci['approval gap'][1] * 100:.0f})")
print(f"  Deleting age: gap {fixes.loc['Age deleted', 'gap (points)']:.1f} points (bias survives through proxies)")
best_fix = "Separate threshold: ratio 0.8, chosen out of fold"
bf = fixes.loc[[i for i in fixes.index if i.startswith(best_fix)][0]]
print(f"  Best fix (cut-off chosen out of fold): gap {bf['gap (points)']:.1f} points, good-risk gap {bf['good-risk gap (points)']:.1f}, cost {bf['cost']:,.0f} (original {base_cost:,.0f})")
print(f"  Flip test: {gained.sum()} of {young.sum()} under-25s would be approved if they were 35")